**📦 STEP 1: IMPORT & LOAD**

In [340]:
import pandas as pd
import numpy as np

calls_df = pd.read_csv("911.csv")
aqi_df = pd.read_csv("aqi.csv")
weather_df = pd.read_csv("weather.csv")
population_df = pd.read_csv("population.csv")

**🚑 STEP 2: EXTRACT EMS PATTERN (911)**

In [341]:
calls_df = calls_df.dropna(subset=["title", "timeStamp"])

calls_df = calls_df[calls_df["title"].str.contains("EMS", na=False)]

calls_df["datetime"] = pd.to_datetime(calls_df["timeStamp"])

calls_df["hour"] = calls_df["datetime"].dt.hour
calls_df["dow"] = calls_df["datetime"].dt.dayofweek

# Hourly demand pattern
hourly_pattern = (
    calls_df.groupby("hour")
    .size()
    .reset_index(name="count")
)

hourly_pattern["weight"] = (
    hourly_pattern["count"] / hourly_pattern["count"].sum()
)

**🌫 STEP 3: CLEAN AQI (DELHI)**

In [342]:
aqi_df = aqi_df[aqi_df["City"] == "Delhi"].copy()

aqi_df["date"] = pd.to_datetime(aqi_df["Date"])
aqi_df = aqi_df.sort_values("date")

aqi_df[["PM2.5", "PM10", "AQI"]] = (
    aqi_df[["PM2.5", "PM10", "AQI"]]
    .ffill()
    .bfill()
)

aqi_df = aqi_df[["date", "PM2.5", "PM10", "AQI"]]

**🌦 STEP 4: CLEAN WEATHER**

In [343]:
# STEP 4: CLEAN WEATHER (FINAL FIX)

weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.normalize()

# Keep only needed columns
weather_df = weather_df[["date", "meantemp", "humidity"]]

# Rename
weather_df = weather_df.rename(columns={
    "meantemp": "temperature"
})

# Sort properly
weather_df = weather_df.sort_values("date")

# INTERPOLATE BEFORE MERGE (CRITICAL)
weather_df["temperature"] = weather_df["temperature"].interpolate()
weather_df["humidity"] = weather_df["humidity"].interpolate()


In [344]:
# FIND COMMON DATE RANGE

start_date = max(
    aqi_df["date"].min(),
    weather_df["date"].min()
)

end_date = min(
    aqi_df["date"].max(),
    weather_df["date"].max()
)

print("COMMON RANGE:", start_date, "→", end_date)

COMMON RANGE: 2015-01-01 00:00:00 → 2017-01-01 00:00:00


In [345]:
# weather_df["date"] = pd.to_datetime(weather_df["date"])

# weather_df = (
#     weather_df
#     .sort_values("date")
#     .set_index("date")
#     .resample("D")
#     .ffill()
#     .reset_index()
# )

# weather_df = weather_df.rename(columns={
#     "meantemp": "temperature"
# })

**👥 STEP 5: EXTRACT DELHI DISTRICTS (REAL POPULATION)**

In [346]:
delhi_pop = population_df[
    population_df["State Name"] == "NCT OF DELHI"
].copy()

# Keep only useful columns
delhi_pop = delhi_pop[[
    "District name",
    "Population"
]]

delhi_pop = delhi_pop.rename(columns={
    "District name": "zone_name"
})

**STEP 6: CREATE REAL FEATURES**

In [347]:
# Population density proxy (normalize)
delhi_pop["population_density"] = (
    delhi_pop["Population"] / delhi_pop["Population"].mean()
)

# Elderly proxy (use literacy or random slight variation)
np.random.seed(42)
delhi_pop["elderly_pct"] = np.random.uniform(0.06, 0.12, len(delhi_pop))

**🗺 STEP 7: CREATE GRID**

In [348]:
date_range = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

hours = list(range(24))

zones = delhi_pop["zone_name"].unique()

grid = pd.MultiIndex.from_product(
    [date_range, hours, zones],
    names=["date", "hour", "zone_name"]
).to_frame(index=False)

**🔗 STEP 8: MERGE FEATURES**

In [349]:
grid = grid.merge(hourly_pattern[["hour", "weight"]], on="hour", how="left")

grid = grid.merge(delhi_pop, on="zone_name", how="left")

**🔥 STEP 9: GENERATE REALISTIC CALLS**

In [350]:
BASE_CALLS = 250

grid["call_count"] = (
    grid["weight"] * BASE_CALLS *
    grid["population_density"] *
    (1 + grid["elderly_pct"])
)

# Add realistic noise
np.random.seed(42)

grid["call_count"] *= np.random.uniform(0.8, 1.2, len(grid))

# Add peak boost
grid.loc[grid["hour"].between(18, 22), "call_count"] *= 1.3

# Final cleanup
grid["call_count"] = grid["call_count"].astype(int)

**🌍 STEP 10: MERGE AQI + WEATHER**

In [351]:
# Normalize dates
grid["date"] = pd.to_datetime(grid["date"]).dt.normalize()
aqi_df["date"] = pd.to_datetime(aqi_df["date"]).dt.normalize()
weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.normalize()

In [352]:
# STEP 10: MERGE AQI
grid = grid.merge(aqi_df, on="date", how="left")

# STEP 10B: MERGE WEATHER
grid = grid.merge(
    weather_df[["date", "temperature", "humidity"]],
    on="date",
    how="left"
)

# Check missing BEFORE filling
print(grid[["temperature", "humidity"]].isnull().sum())

temperature    0
humidity       0
dtype: int64


In [353]:
# STEP 10C: ADD REALISTIC WEATHER VARIATION

np.random.seed(42)

# Add small natural noise
grid["temperature"] += np.random.normal(0, 1.5, len(grid))
grid["humidity"] += np.random.normal(0, 3, len(grid))

# Add seasonal effect
grid["month"] = grid["date"].dt.month

# Winter (colder)
grid.loc[grid["month"].isin([12, 1, 2]), "temperature"] -= np.random.uniform(3, 6)

# Summer (hotter)
grid.loc[grid["month"].isin([5, 6, 7]), "temperature"] += np.random.uniform(3, 6)

# Clip to realistic bounds
grid["temperature"] = grid["temperature"].clip(5, 45)
grid["humidity"] = grid["humidity"].clip(10, 100)

**⚙ STEP 11: FEATURE ENGINEERING**

In [354]:
grid["day_of_week"] = grid["date"].dt.dayofweek

grid["is_peak_hour"] = grid["hour"].apply(
    lambda x: 1 if 6 <= x <= 9 or 17 <= x <= 21 else 0
)

**🧹 STEP 12: HANDLE MISSING**

In [355]:
# AQI fill
grid["PM2.5"] = grid["PM2.5"].ffill()
grid["PM10"] = grid["PM10"].ffill()
grid["AQI"] = grid["AQI"].ffill()


In [356]:
# 🚨 REMOVE UNREALISTIC EXTREMES

grid = grid[
    (grid["temperature"] >= 8) & (grid["temperature"] <= 42)
]

grid = grid[
    (grid["humidity"] >= 20) & (grid["humidity"] <= 95)
]

In [357]:
# 🎯 ADD SMALL NOISE (not chaos)

np.random.seed(42)

grid["temperature"] += np.random.normal(0, 0.8, len(grid))
grid["humidity"] += np.random.normal(0, 2, len(grid))

In [358]:
print(grid[["temperature", "humidity"]].isnull().sum())

temperature    0
humidity       0
dtype: int64


In [359]:
print(grid[["temperature", "humidity"]].describe())

         temperature       humidity
count  141588.000000  141588.000000
mean       26.770429      59.841124
std         9.285141      15.345211
min         5.522621      14.634712
25%        20.238389      50.685670
50%        28.761169      60.807673
75%        33.617965      70.370294
max        44.405238     100.774853


In [360]:
grid.isnull().sum()

,0
date,0
hour,0
zone_name,0
weight,0
Population,0
population_density,0
elderly_pct,0
call_count,0
PM2.5,0
PM10,0


**💾 STEP 13: FINAL DATASET**

In [361]:
final_df = grid[
    [
        "date",
        "hour",
        "zone_name",
        "call_count",
        "PM2.5",
        "PM10",
        "AQI",
        "temperature",
        "humidity",
        "population_density",
        "elderly_pct",
        "day_of_week",
        "is_peak_hour"
    ]
]

In [362]:
final_df.sample(10)

,date,hour,zone_name,call_count,PM2.5,PM10,AQI,temperature,humidity,population_density,elderly_pct,day_of_week,is_peak_hour
99430,2016-04-05,7,South West,14,97.83,174.11,261.0,28.947385,30.731416,1.229253,0.111971,1,1
85434,2016-01-31,12,West,18,242.05,291.16,437.0,11.686757,78.595505,1.363430,0.063485,6,0
106433,2016-05-07,17,South,18,77.89,193.14,229.0,39.049171,36.964611,1.464585,0.096067,5,1
57551,2015-09-24,10,Central,4,63.06,187.90,175.0,27.527485,58.777381,0.312181,0.069360,3,0
87228,2016-02-08,20,North West,35,119.57,186.86,301.0,11.748824,72.071644,1.960267,0.082472,0,1
22054,2015-04-13,2,New Delhi,0,86.49,116.62,247.0,26.442423,70.403833,0.076128,0.069361,0,0
937,2015-01-05,8,North,7,146.60,219.13,325.0,7.890892,71.763782,0.476044,0.117043,0,1
82297,2016-01-17,0,North,3,179.48,268.50,321.0,8.400775,69.749336,0.476044,0.117043,6,0
118845,2016-07-04,5,North West,13,57.10,75.83,160.0,38.062149,74.620026,1.960267,0.082472,0,0
116275,2016-06-22,7,New Delhi,0,75.14,99.89,187.0,39.958520,66.858649,0.076128,0.069361,2,1


In [363]:
print(final_df["temperature"].nunique())
print(final_df["humidity"].nunique())

141588
141588


In [364]:
print(final_df["temperature"].value_counts().head(10))
print(final_df["humidity"].value_counts().head(10))

temperature
10.593157    1
10.351299    1
8.890849     1
10.698540    1
7.419685     1
8.010249     1
10.977705    1
11.242475    1
10.108436    1
9.437984     1
Name: count, dtype: int64
humidity
88.887857    1
68.699609    1
73.110841    1
66.802684    1
82.306403    1
91.523271    1
93.461489    1
79.422065    1
89.307349    1
84.258157    1
Name: count, dtype: int64


In [365]:
final_df.shape

(141588, 13)

In [366]:
final_df["temperature"] = final_df["temperature"].round(2)
final_df["humidity"] = final_df["humidity"].round(2)
final_df["elderly_pct"] = final_df["elderly_pct"].round(2)
final_df["population_density"] = final_df["population_density"].round(2)

final_df.sample(3)

/tmp/ipykernel_2708/2340696351.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["temperature"] = final_df["temperature"].round(2)
/tmp/ipykernel_2708/2340696351.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["humidity"] = final_df["humidity"].round(2)
/tmp/ipykernel_2708/2340696351.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

,date,hour,zone_name,call_count,PM2.5,PM10,AQI,temperature,humidity,population_density,elderly_pct,day_of_week,is_peak_hour
53303,2015-09-04,18,Central,5,97.31,284.54,368.0,34.01,44.64,0.31,0.07,4,1
101384,2016-04-14,8,South,16,94.81,220.48,239.0,32.72,27.20,1.46,0.10,3,1
60377,2015-10-07,12,Central,5,144.04,305.35,383.0,30.15,65.58,0.31,0.07,2,0


**📥 STEP 14: EXPORT**

In [367]:
final_df.to_csv("ambucast_data.csv", index=False)